## Import libraries

In [ ]:
import os, time

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython import display

import tensorflow as tf
from tensorflow.keras.layers import Input, Reshape, Dropout, Dense 
from tensorflow.keras.layers import Flatten, BatchNormalization
from tensorflow.keras.layers import Activation, ZeroPadding2D
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import UpSampling2D, Conv2D, Conv2DTranspose
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.optimizers import Adam

In [ ]:
# Convert seconds to HH:MM:SS.MI format
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)

In [ ]:
IMAGE_CHANNELS = 3
GENERATE_RESOLUTION_FACTOR = 4
GENERATE_SQUARE = 32 * GENERATE_RESOLUTION_FACTOR

# Preview image details
PREVIEW_ROWS, PREVIEW_COLS = 4, 7
PREVIEW_MARGIN = 8

# Source vector size to generate images
SEED_SIZE = 100

# Model configuration
DATA_PATH = '/kaggle/input/cartoon-faces-googles-cartoon-set'
FOLDER_NAME = 'cartoonset100k_jpg'

EPOCHS = 80
BATCH_SIZE = 64
BUFFER_SIZE = 61000

print(f"Image details: {GENERATE_SQUARE}px")

In [ ]:
EPOCHS = 60

# Load the Dataset

We have a lot of images to read and process and this task can take a while. So, we will real all the iamges, convert it to JPG format, resize it, normalize it and store the processed file as binary.

We are using Numpy binary instead of Pickle because the file is very large and may cause problems with some versions of Pickle.

In [ ]:
!rm /kaggle/working/training_data_cartoonset100k_jpg128128.npy

In [ ]:
import random

# Define the training binary path
training_binary_path = os.path.join(
    "/kaggle/working", 
    f'training_data_{FOLDER_NAME}{GENERATE_SQUARE}{GENERATE_SQUARE}.npy'
)
print("Binary file name: ", training_binary_path)

if not os.path.isfile(training_binary_path):
    start = time.time()
    print("File not found, loading training images...")

    training_data = []
    faces_path = os.path.join(DATA_PATH, FOLDER_NAME)
    
    # Check if the folder exists
    if not os.path.exists(faces_path):
        raise FileNotFoundError(f"The folder path {faces_path} does not exist.")
    
    # List all subdirectories (0, 1, 2, ..., 9) and shuffle them
    subfolders = os.listdir(faces_path)
    random.shuffle(subfolders)  # Shuffle the subfolder list for randomness
    print("Shuffled subfolders (categories):", subfolders)
    
    for subfolder in tqdm(subfolders):
        if len(training_data) >= 35000: 
            break
        subfolder_path = os.path.join(faces_path, subfolder)
        
        # Ensure the current item is a directory
        if os.path.isdir(subfolder_path):
            # List all files in the subfolder
            image_files = os.listdir(subfolder_path)
            random.shuffle(image_files)  # Shuffle the files for randomness
            
            for filename in image_files:
                path = os.path.join(subfolder_path, filename)
                
                # Process only image files (filter extensions)
                if path.lower().endswith(('.jpg', '.jpeg', '.png')):
                    try:
                        image = Image.open(path).convert("RGB").resize(
                            (GENERATE_SQUARE, GENERATE_SQUARE))
                        if len(training_data) >= 35000: 
                            break
                        training_data.append(np.asarray(image))
                    except Exception as e:
                        print(f'[Error] Unable to process image at path {path}, exception: {e}')
                        continue  # Skip problematic images and continue

    # Reshape, normalize, and convert to NumPy array
    if len(training_data) == 0:
        raise ValueError("No valid images were found in the dataset.")
    
    training_data = np.reshape(
        training_data, (-1, GENERATE_SQUARE, GENERATE_SQUARE, IMAGE_CHANNELS)
    )
    training_data = training_data.astype(np.float32)
    training_data = training_data / 127.5 - 1.  # Normalize to [-1, 1]

    # Save the training data as a binary file for future use
    print("Saving training image binary file...")
    np.save(training_binary_path, training_data)
    stop = time.time() - start
    print(f'Image preprocess time: {stop:.2f} seconds')
else:
    print("Loading previous training binary file...")
    load_data_start_time = time.time()
    training_data = np.load(training_binary_path)
    load_data_tat = time.time() - load_data_start_time
    print(f"Data load time: {load_data_tat:.2f} seconds")


In [ ]:
# Shuffle the dataset and create batches
train_dataset = tf.data.Dataset.from_tensor_slices(training_data).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
print("Length of total images: ", len(training_data))
print("Length of total batches: ", len(train_dataset))
print("Length of images in one batch: ", len(list(train_dataset.as_numpy_iterator())))

## Define the Models

We will use [Keras Sequential API](https://www.tensorflow.org/guide/keras#sequential_model) to define the models.

### Generator

The generator needs upsampling layers to produce an image from random noise. We start with a `Dense` layer whose input is seed and then apply [`tf.keras.layers.Conv2DTranspose`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose) to upsample it several times until we reach the output shape of `96x96x3`. We are using `LeakyReLU` activation for middle layers and `tanh` for output layer.

In [ ]:
def build_generator(seed_size, channels):
    model = Sequential()

    # Initial dense layer
    model.add(Dense(4 * 4 * 256, activation="relu", input_dim=seed_size))
    model.add(BatchNormalization())
    model.add(LeakyReLU())
    model.add(Reshape((4, 4, 256)))

    # First upsampling: 4x4 -> 8x8
    model.add(Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 8, 8, 256)
    model.add(BatchNormalization())
    model.add(LeakyReLU())

    # Second upsampling: 8x8 -> 16x16
    model.add(Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 16, 16, 256)
    model.add(BatchNormalization())
    model.add(LeakyReLU())

    # Third upsampling: 16x16 -> 32x32
    model.add(Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 128)
    model.add(BatchNormalization())
    model.add(LeakyReLU())

    # Fourth upsampling: 32x32 -> 64x64
    model.add(Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 64, 64, 64)
    model.add(BatchNormalization())
    model.add(LeakyReLU())

    # Fifth upsampling: 64x64 -> 128x128
    model.add(Conv2DTranspose(channels, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 128, 128, channels)

    return model


### The Discriminator

Discriminator is a CNN image classifier.

In [ ]:
def build_discriminator(image_shape):
    model = Sequential()

    model.add(Conv2D(32, kernel_size=3, strides=2, input_shape=image_shape, 
                     padding="same"))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Dropout(0.2))
    model.add(Conv2D(64, kernel_size=3, strides=2, padding="same"))
    model.add(ZeroPadding2D(padding=((0,1),(0,1))))
    model.add(BatchNormalization(momentum=0.7))
    model.add(LeakyReLU(alpha=0.3))

    model.add(Dropout(0.2))
    model.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Dropout(0.2))
    model.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.7))
    model.add(LeakyReLU(alpha=0.3))

    model.add(Dropout(0.2))
    model.add(Conv2D(512, kernel_size=3, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    return model

Save the generated images during training to check the training progress.

In [ ]:
def save_images(cnt, noise):
    GRID_SPACING = 16  # Dynamic spacing for the preview grid
    image_array = np.full(( 
        PREVIEW_MARGIN + (PREVIEW_ROWS * (GENERATE_SQUARE + GRID_SPACING)), 
        PREVIEW_MARGIN + (PREVIEW_COLS * (GENERATE_SQUARE + GRID_SPACING)), 3), 
        255, dtype=np.uint8)

    generated_images = generator.predict(noise)
    generated_images = 0.5 * generated_images + 0.5  # Rescale to [0, 1]

    image_count = 0
    for row in range(PREVIEW_ROWS):
        for col in range(PREVIEW_COLS):
            r = row * (GENERATE_SQUARE + GRID_SPACING) + PREVIEW_MARGIN
            c = col * (GENERATE_SQUARE + GRID_SPACING) + PREVIEW_MARGIN
            image_array[r:r + GENERATE_SQUARE, c:c + GENERATE_SQUARE] = (
                generated_images[image_count] * 255
            )
            image_count += 1

    output_path = os.path.join("/kaggle/working", 'output')
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    filename = os.path.join(output_path, f"train-{cnt}.png")
    im = Image.fromarray(image_array)

    fig = plt.figure(figsize=(15, 15))
    plt.imshow(im)
    plt.axis("off")  # Hide axis for better visualization
    plt.show()
    im.save(filename)

Lets plot the image created by untrained Generator.

In [ ]:
generator = build_generator(SEED_SIZE, IMAGE_CHANNELS)

noise = tf.random.normal([1, SEED_SIZE])
generated_image = generator(noise, training=False)
print("Generated image shape: ", generated_image.shape)

plt.imshow(generated_image[0, :, :, 0])

In [ ]:
image_shape = (GENERATE_SQUARE,GENERATE_SQUARE,IMAGE_CHANNELS)

discriminator = build_discriminator(image_shape)
decision = discriminator(generated_image)
print (decision)

## Define the loss and optimizers

We will use the Binary Cross-Entropy loss function. BCE cost function has two parts, one relevant for each class. The value is close to zero when the label and prediction are similar but approaches infinity when the label and the prediction are different.

In [ ]:
# This method returns a helper function to compute cross entropy loss
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

Both Generator and Discriminator model uses Adam optimizer and the same learning rate and momentum.

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1.5e-4,0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(1.5e-4,0.5)

## Training Pipeline

The training function is decorated by `tf.function annotation` which causes the function to be precompiled and improves performance.

In [ ]:
# Notice the use of `tf.function`
# This annotation causes the function to be "compiled".
@tf.function
def train_step(images):
  seed = tf.random.normal([BATCH_SIZE, SEED_SIZE])

  with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
    generated_images = generator(seed, training=True)

    real_output = discriminator(images, training=True)
    fake_output = discriminator(generated_images, training=True)

    gen_loss = generator_loss(fake_output)
    disc_loss = discriminator_loss(real_output, fake_output)
    

    gradients_of_generator = gen_tape.gradient(\
        gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(\
        disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(
        gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(
        gradients_of_discriminator, 
        discriminator.trainable_variables))
  # generate_and_save_images(generator, epochs, seed)
  return gen_loss,disc_loss

In [ ]:
def generate_and_save_images(model, epoch, test_input):
  # Notice `training` is set to False.
  # This is so all layers run in inference mode (batchnorm).
  predictions = model(test_input, training=False)

  fig = plt.figure(figsize=(4,4))

  for i in range(predictions.shape[0]):
      plt.subplot(4, 4, i+1)
      plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
      plt.axis('off')

  plt.savefig('image_at_epoch_{:04d}.png'.format(epoch))
  plt.show()

Train the two neural networks independently in two separate passes.

In [ ]:
def train(dataset, epochs):
  fixed_seed = np.random.normal(0, 1, (PREVIEW_ROWS * PREVIEW_COLS, 
                                       SEED_SIZE))
  start = time.time()

  for epoch in range(epochs):
    epoch_start = time.time()

    gen_loss_list = []
    disc_loss_list = []

    for image_batch in dataset:
      t = train_step(image_batch)
      gen_loss_list.append(t[0])
      disc_loss_list.append(t[1])

    g_loss = sum(gen_loss_list) / len(gen_loss_list)
    d_loss = sum(disc_loss_list) / len(disc_loss_list)

    epoch_elapsed = time.time()-epoch_start
    print (f'Epoch {epoch+1}, gen loss={g_loss},disc loss={d_loss}, time={hms_string(epoch_elapsed)}')
    
    display.clear_output(wait=True)
    print (f'Epoch {epoch+1}, gen loss={g_loss},disc loss={d_loss}, time={hms_string(epoch_elapsed)}')
    save_images(epoch,fixed_seed)

  display.clear_output(wait=True)
  print (f'Epoch {epoch+1}, gen loss={g_loss},disc loss={d_loss}, time={hms_string(epoch_elapsed)}')
  save_images(epoch,fixed_seed)

  elapsed = time.time()-start
  print (f'Training time: {hms_string(elapsed)}')

In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
name = f"face_generator_{EPOCHS}.h5"
generator.save(os.path.join("/kaggle/working", name))

In [ ]:
from tensorflow.keras.models import load_model


In [ ]:

# Load the generator generator
testmodel = load_model('/kaggle/working/face_generator_50.h5')

In [ ]:
SEED_SIZE = 100

noise = tf.random.normal([1, SEED_SIZE])
# Generate the image

In [ ]:
generated_image.shape

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt


# Constants
SEED_SIZE = 100  # The same size used during training
IMAGE_SIZE = 96  # Generated image dimensions

# Load the generator model
testmodel = load_model("/kaggle/working/face_generator_60.h5")  # Update with your saved model path

In [ ]:
# Function to generate images
def generate_images(generator, num_images=16):
    # Generate random noise
    noise = tf.random.normal([num_images, SEED_SIZE])
    
    # Generate images
    generated_images = testmodel.predict(noise)
    
    # Rescale pixel values from [-1, 1] to [0, 1]
    generated_images = (generated_images + 1) / 2.0
    
    # Plot the generated images
    plt.figure(figsize=(10, 10))
    for i in range(num_images):
        plt.subplot(4, 4, i + 1)
        plt.imshow(generated_images[i])
        plt.axis('off')
    plt.show()

In [ ]:
# Generate and display images
generate_images(generator, num_images=1)

In [1]:
%%capture
!pip install colabcode
!pip install fastapi

In [2]:
!ngrok config add-authtoken 2p3dSMQUeAkkflqWBjto6Hui80Z_qPjMnnUFpb3q2cCD9wwp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml                                


In [3]:
from colabcode import ColabCode
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import os
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image
import tensorflow as tf

app = FastAPI(title="DCGAN Backend API", description="with FastAPI and ColabCode", version="1.0")

cc = ColabCode(port=12000, code=False)

# Load the generator model
small_generator = load_model('/kaggle/working/face_generator_60.h5')
large_generator=load_model('/kaggle/working/face_generator_80.h5')
# Constants
SEED_SIZE = 100  # The same size used during training
IMAGE_SIZE = 96  # Generated image dimensions

# Enable CORS for localhost
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "pixelperson.classikh.me"],  
    allow_credentials=True,
    allow_methods=["*"],  
    allow_headers=["*"],  
)

@app.get("/health")
async def read_root():
    return {"message": "okay"}

from fastapi import Query

from fastapi import Query


@app.get("/generate-image")
async def generate_image(
    transparent: bool = Query(default=False),
    size: str = Query(default="small", pattern="^(small|large)$")
):
    generator = large_generator if size == "large" else small_generator
    
    noise = tf.random.normal([1, SEED_SIZE])
    
    # Generate the image
    generated_image = generator.predict(noise)
    
    # Rescale pixel values from [-1, 1] to [0, 255]
    generated_image = ((generated_image[0] + 1) * 127.5).astype(np.uint8)
    
    # Convert to an image
    image = Image.fromarray(generated_image)

    if transparent:
        # Convert to RGBA and make the background transparent
        image = image.convert("RGBA")
        data = image.getdata()
        new_data = []
        for item in data:
            # Check for white pixels
            if item[0] > 240 and item[1] > 240 and item[2] > 240:
                new_data.append((255, 255, 255, 0))  # Transparent pixel
            else:
                new_data.append(item)
        image.putdata(new_data)
        
        # Save the transparent image
        output_path = f"/kaggle/working/generated_image_transparent_{size}.png"
    else:
        # Save the normal image
        output_path = f"/kaggle/working/generated_image_{size}.png"

    # Save the image
    image.save(output_path, "PNG")

    # Return the image as a response
    return FileResponse(output_path, media_type="image/png")


In [ ]:
cc.run_app(app=app)

INFO:     Started server process [30]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:12000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://607e-35-236-140-2.ngrok-free.app" -> "http://localhost:12000"
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "OPTIONS /generate-image HTTP/1.1" 200 OK


I0000 00:00:1732174990.387170     133 service.cc:145] XLA service 0x7ae0d0003740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1732174990.387236     133 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image HTTP/1.1" 200 OK


I0000 00:00:1732174991.958921     133 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "OPTIONS /generate-image%26size%3Dlarge HTTP/1.1" 200 OK
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image%26size%3Dlarge HTTP/1.1" 404 Not Found
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "OPTIONS /generate-image%26transparent%3Dtrue%26size%3Dlarge HTTP/1.1" 200 OK
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image%26transparent%3Dtrue%26size%3Dlarge HTTP/1.1" 404 Not Found
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "OPTIONS /generate-image%26transparent%3Dtrue HTTP/1.1" 200 OK
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image%26transparent%3Dtrue HTTP/1.1" 404 Not Found
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image%26transparent%3Dtrue HTTP/1.1" 404 Not Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
INFO:     2401:4900:5c7f:9cd3:aab7:19c0:df57:8305:0 - "GET /generate-image HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━